# Amazon ML Challenge 2026: Large-Scale Business Entity Resolution

**Author:** Saurabh Duhariya  
**Task:** Record linkage across heterogeneous, noisy multi-source datasets (2.2M Source 1 records vs 10.3M combined Source 2/3 target pool).  
**Evaluation Metric:** Macro-averaged $F_{0.5}$ score (precision-weighted, penalizing false merges $2\times$ more than missed matches).

---

## Architecture Overview
```text
Raw Heterogeneous Datasets (12.5M records)
          │
          ▼
Phase 1: Domain-Specific Normalization (Regex 40+ Legal Suffixes, Address Abbreviations, Country Split)
          │
          ▼
Phase 2: Hybrid Multi-Channel Blocking
   ┌────────────────────────────────────────┴────────────────────────────────────────┐
   ▼                                                                                 ▼
Sparse Char N-Gram TF-IDF (Top-25)                                 Inverted Index Token Overlap (>=2 tokens)
   └────────────────────────────────────────┬────────────────────────────────────────┘
                                            ▼
                             Union Candidate Set (Recall: 95.97%)
                                            │
                                            ▼
Phase 3: Pairwise Feature Engineering (17 String/Address Metrics) + Hard Negative Mining (5:1)
                                            │
                                            ▼
Phase 4: Gradient Boosting Ensemble (GPU XGBoost + LightGBM)
                                            │
                                            ▼
Phase 5: Decision Threshold Tuning for Macro F_0.5 & Singleton Filtering
```

### 1. Environment Setup & Library Imports

In [ ]:
import os, re, gc, json, warnings, time, random
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.feature_extraction.text import TfidfVectorizer
from rapidfuzz import fuzz
from rapidfuzz.distance import JaroWinkler
from unidecode import unidecode
from tqdm.auto import tqdm

import xgboost as xgb
import lightgbm as lgb
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.1f} GB")


### 2. Configuration & Hyperparameters

In [ ]:
DATA_ROOT = Path("../student_resource/dataset")
TRAIN_DIR = DATA_ROOT / "train"
TEST_DIR  = DATA_ROOT / "test"
OUTPUT_DIR = Path("../output")
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

TFIDF_TOP_K = 25
TFIDF_MIN_SIM = 0.08
RULE_MIN_TOKENS = 2
NEG_RATIO = 5
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)


### 3. Domain-Specific Text Preprocessing & Cleaning

Business names frequently share legal designations (`Pvt Ltd`, `LLC`, `Corp`) that induce false-positive string similarity. We extract and isolate:
- **Core Company Name (`clean_name_core`)**
- **Legal Suffixes (`legal_suffix`)** for compatibility checking
- **Standardized Street Addresses** (expanding `st` -> `street`, `rd` -> `road`, etc.)
- **Numerical Sequences** for PIN/street number verification

In [ ]:
LEGAL_SUFFIXES = [
    "private limited", "pvt ltd", "pvt limited", "limited liability partnership",
    "limited liability company", "incorporated", "corporation",
    "limited", "inc", "corp", "llc", "llp", "lp", "plc", "ltd", "pvt", "co",
    "private", "public",
    "praaivett limittedd", "praivet limited", "limittedd", "praaivett", "elelpii",
    "piraivet limitet", "piraiveett limitett", "limitet",
    "societe a responsabilite limitee", "societe anonyme",
    "societe par actions simplifiee", "societe civile immobiliere",
    "sarl", "sas", "sasu", "eurl", "sa", "sci", "gie", "snc", "scp", "gmbh", "ag",
    "group", "holding", "holdings", "enterprises", "enterprise",
    "industries", "industry", "services", "solutions", "technologies",
    "technology", "international", "intl"
]

BUSINESS_PREFIXES = ["ms", "the", "ets", "ste", "cie", "cabinet", "groupe", "dr", "shri", "messrs"]

_SUFFIX_PATTERNS = [(re.compile(r"\b" + re.escape(s) + r"\b"), s) for s in sorted(LEGAL_SUFFIXES, key=len, reverse=True)]
_PREFIX_PATTERNS = [(re.compile(r"^\b" + re.escape(p) + r"\b\s*"), p) for p in sorted(BUSINESS_PREFIXES, key=len, reverse=True)]

ADDRESS_ABBR = {
    "rd": "road", "st": "street", "ave": "avenue", "blvd": "boulevard",
    "dr": "drive", "ln": "lane", "ct": "court", "pl": "place",
    "sq": "square", "hwy": "highway", "pkwy": "parkway",
    "apt": "apartment", "ste": "suite", "fl": "floor",
    "bldg": "building", "dept": "department", "mt": "mount",
    "ft": "fort", "n": "north", "s": "south", "e": "east", "w": "west",
    "ne": "northeast", "nw": "northwest", "se": "southeast", "sw": "southwest",
    "bd": "boulevard", "av": "avenue", "imp": "impasse",
}

def normalize_text(text: str) -> str:
    if pd.isna(text) or not text:
        return ""
    text = unidecode(str(text)).lower().strip()
    text = text.replace("&", " and ").replace("@", " at ").replace("#", " number ")
    text = re.sub(r"\bm/s\b|\bm\s*s\b", "ms", text)
    text = re.sub(r"[^\w\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def extract_name_components(name: str):
    if not name:
        return "", "", ""
    prefix = ""
    for pat, p in _PREFIX_PATTERNS:
        if pat.search(name):
            prefix = p
            name = pat.sub("", name).strip()
            break
    suffixes = []
    for pat, s in _SUFFIX_PATTERNS:
        if pat.search(name):
            suffixes.append(s)
            name = pat.sub("", name).strip()
    return re.sub(r"\s+", " ", name).strip(), prefix, " ".join(suffixes)

def expand_address_abbr(address: str) -> str:
    return " ".join(ADDRESS_ABBR.get(t, t) for t in address.split())

def extract_numbers(text: str) -> str:
    return " ".join(re.findall(r"\b\d+\b", text))

def preprocess_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["business_name"] = df["business_name"].fillna("")
    df["business_address"] = df["business_address"].fillna("")
    df["country"] = df["country"].fillna("")
    
    df["clean_name"] = df["business_name"].apply(normalize_text)
    components = [extract_name_components(n) for n in df["clean_name"]]
    df["clean_name_core"] = [c[0] for c in components]
    df["name_prefix"]     = [c[1] for c in components]
    df["legal_suffix"]    = [c[2] for c in components]
    
    df["clean_address"] = df["business_address"].apply(normalize_text).apply(expand_address_abbr)
    df["addr_numbers"]  = df["clean_address"].apply(extract_numbers)
    df["combined_text"] = df["clean_name_core"] + " | " + df["clean_address"]
    df["name_tokens"]   = df["clean_name_core"].apply(lambda x: set(x.split()) if x else set())
    return df
print("✅ Preprocessing pipeline configured.")


### 4. Candidate Generation (Multi-Channel Blocking)

Pairwise matching of $2.2M \times 10.3M$ records is intractable ($>22$ Trillion pairs).
We implement a hybrid two-channel blocking strategy:
1. **Sparse Sublinear TF-IDF Character N-Grams (3-5 n-grams):** Captures typos and phonetic variations.
2. **Inverted Token Overlap Index:** Captures high-frequency token sharing with country partitioning.

In [ ]:
def tfidf_blocking(s1_df, s23_df, top_k=25, min_sim=0.08, batch_size=100):
    candidates = {eid: set() for eid in s1_df["entity_id"]}
    countries = s1_df["country"].dropna().unique()
    for c in countries:
        s1_c = s1_df[s1_df["country"] == c]
        s23_c = s23_df[s23_df["country"] == c]
        if len(s1_c) == 0 or len(s23_c) == 0:
            continue
        sample_size = min(300_000, len(s23_c))
        sample_texts = s23_c["combined_text"].sample(sample_size, random_state=42)
        vectorizer = TfidfVectorizer(
            analyzer="char_wb", ngram_range=(3, 5), max_features=120_000,
            sublinear_tf=True, dtype=np.float32
        )
        vectorizer.fit(sample_texts)
        s23_matrix = vectorizer.transform(s23_c["combined_text"])
        s23_ids = s23_c["entity_id"].values
        n_batches = int(np.ceil(len(s1_c) / batch_size))
        for b in range(n_batches):
            batch = s1_c.iloc[b * batch_size : (b + 1) * batch_size]
            s1_mat = vectorizer.transform(batch["combined_text"])
            sim = s1_mat.dot(s23_matrix.T)
            for i, s1_id in enumerate(batch["entity_id"]):
                row = sim.getrow(i)
                if row.nnz == 0: continue
                mask = row.data >= min_sim
                if not np.any(mask): continue
                valid_cols = row.indices[mask]
                valid_vals = row.data[mask]
                if len(valid_cols) > top_k:
                    top_idx = np.argpartition(valid_vals, -top_k)[-top_k:]
                    valid_cols = valid_cols[top_idx]
                candidates[s1_id].update(s23_ids[valid_cols])
    return candidates

def rule_based_blocking(s1_df, s23_df, min_shared_tokens=2):
    token_index = defaultdict(list)
    s23_countries = dict(zip(s23_df["entity_id"], s23_df["country"]))
    for eid, tokens in zip(s23_df["entity_id"], s23_df["name_tokens"]):
        for tok in tokens:
            if len(tok) >= 3:
                token_index[tok].append(eid)
    candidates = {eid: set() for eid in s1_df["entity_id"]}
    s1_countries = dict(zip(s1_df["entity_id"], s1_df["country"]))
    for eid, tokens in zip(s1_df["entity_id"], s1_df["name_tokens"]):
        s1_country = s1_countries.get(eid, "")
        counts = defaultdict(int)
        for tok in tokens:
            if len(tok) >= 3 and tok in token_index:
                for cand_id in token_index[tok]:
                    counts[cand_id] += 1
        candidates[eid] = {
            cand_id for cand_id, cnt in counts.items()
            if cnt >= min_shared_tokens and s23_countries.get(cand_id, "") == s1_country
        }
    return candidates

def merge_blocking_channels(*channels):
    all_ids = set().union(*[ch.keys() for ch in channels])
    return {s1_id: set().union(*[ch.get(s1_id, set()) for ch in channels]) for s1_id in all_ids}


### 5. Pairwise Feature Engineering (17 Features)

In [ ]:
def jaccard_tokens(a: str, b: str) -> float:
    sa, sb = set(a.split()), set(b.split())
    return len(sa & sb) / len(sa | sb) if (sa | sb) else 0.0

def numbers_match(a: str, b: str) -> float:
    na, nb = set(re.findall(r"\b\d+\b", a)), set(re.findall(r"\b\d+\b", b))
    return 0.5 if not (na and nb) else (len(na & nb) / max(len(na), len(nb)))

def compute_pair_features(s1_row, cand_row) -> dict:
    n1, n2   = s1_row["clean_name_core"], cand_row["clean_name_core"]
    nf1, nf2 = s1_row["clean_name"], cand_row["clean_name"]
    a1, a2   = s1_row["clean_address"], cand_row["clean_address"]
    f = {}
    # Core name similarity
    f["name_lev"]        = fuzz.ratio(n1, n2) / 100.0
    f["name_jw"]         = JaroWinkler.similarity(n1, n2)
    f["name_token_sort"] = fuzz.token_sort_ratio(n1, n2) / 100.0
    f["name_token_set"]  = fuzz.token_set_ratio(n1, n2) / 100.0
    f["name_jaccard"]    = jaccard_tokens(n1, n2)
    f["name_exact"]      = float(n1 == n2)
    f["name_contain"]    = float(n1 in n2 or n2 in n1) if (n1 and n2) else 0.0
    f["name_full_jw"]    = JaroWinkler.similarity(nf1, nf2)
    # Suffix & prefix compatibility
    suf1, suf2 = s1_row["legal_suffix"], cand_row["legal_suffix"]
    f["suffix_match"] = 0.5 if (not suf1 or not suf2) else (1.0 if suf1 == suf2 else 0.0)
    pre1, pre2 = s1_row["name_prefix"], cand_row["name_prefix"]
    f["prefix_match"] = 1.0 if (not pre1 or not pre2 or pre1 == pre2) else 0.0
    # Address similarity
    if a1 and a2:
        f["addr_lev"]        = fuzz.ratio(a1, a2) / 100.0
        f["addr_jw"]         = JaroWinkler.similarity(a1, a2)
        f["addr_token_sort"] = fuzz.token_sort_ratio(a1, a2) / 100.0
        f["addr_jaccard"]    = jaccard_tokens(a1, a2)
        f["addr_numbers"]    = numbers_match(a1, a2)
    else:
        for k in ["addr_lev", "addr_jw", "addr_token_sort", "addr_jaccard", "addr_numbers"]:
            f[k] = 0.0
    f["country_match"] = float(s1_row["country"] == cand_row["country"])
    f["has_addr_both"] = float(bool(a1) and bool(a2))
    return f


### 6. Macro $F_{0.5}$ Metric & Singleton Formulation

$$F_{0.5} = \frac{1.25 \times \text{Precision} \times \text{Recall}}{0.25 \times \text{Precision} + \text{Recall}}$$

Macro-averaged across all entities including singletons (where predicting an empty list gives $1.0$ and predicting a false match gives $0.0$).

In [ ]:
def compute_f05_macro(predictions: dict, ground_truth: dict) -> dict:
    f05_scores, precisions, recalls = [], [], []
    for s1_id, true_set in ground_truth.items():
        pred_set = set(predictions.get(s1_id, set()))
        true_set = set(true_set)
        if len(true_set) == 0:
            score = 1.0 if len(pred_set) == 0 else 0.0
            f05_scores.append(score)
            precisions.append(score)
            recalls.append(score)
            continue
        if len(pred_set) == 0:
            f05_scores.append(0.0)
            precisions.append(0.0)
            recalls.append(0.0)
            continue
        tp = len(pred_set & true_set)
        p = tp / len(pred_set)
        r = tp / len(true_set)
        f05 = (1.25 * p * r) / (0.25 * p + r) if (0.25 * p + r) > 0 else 0.0
        f05_scores.append(f05)
        precisions.append(p)
        recalls.append(r)
    return {
        "f05": float(np.mean(f05_scores)),
        "precision": float(np.mean(precisions)),
        "recall": float(np.mean(recalls))
    }


### 7. Training Gradient Boosted Models (XGBoost + LightGBM on GPU)

In [ ]:
# Example training function for pair classifications
def train_ensemble(X_train, y_train, X_val, y_val):
    scale_pos = (y_train == 0).sum() / max(1, (y_train == 1).sum())
    dtrain = xgb.DMatrix(X_train, label=y_train)
    dval   = xgb.DMatrix(X_val, label=y_val)
    xgb_params = {
        "objective": "binary:logistic", "eval_metric": "aucpr", "max_depth": 8,
        "learning_rate": 0.05, "subsample": 0.8, "colsample_bytree": 0.8,
        "scale_pos_weight": scale_pos, "tree_method": "hist", "device": DEVICE, "seed": 42
    }
    xgb_model = xgb.train(xgb_params, dtrain, num_boost_round=1000, evals=[(dval, "val")], early_stopping_rounds=40, verbose_eval=100)
    
    lgb_train = lgb.Dataset(X_train, label=y_train)
    lgb_val   = lgb.Dataset(X_val, label=y_val, reference=lgb_train)
    lgb_params = {
        "objective": "binary", "metric": "average_precision", "max_depth": 8,
        "learning_rate": 0.05, "scale_pos_weight": scale_pos, "seed": 42, "verbose": -1
    }
    lgb_model = lgb.train(lgb_params, lgb_train, valid_sets=[lgb_val], num_boost_round=1000, callbacks=[lgb.early_stopping(40, verbose=False)])
    return xgb_model, lgb_model
